In [ ]:
# Import thư viện
from pathlib import Path

import pandas as pd

In [ ]:
# Đọc dữ liệu raw
project_root = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
raw_file = project_root / "data" / "raw" / "hanoi_weather_2019_2025.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_file)
df["datetime"] = pd.to_datetime(df["datetime"])
df = df.sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

df.head()

In [ ]:
# Tạo đặc trưng thời gian
df["hour"] = df["datetime"].dt.hour
df["month"] = df["datetime"].dt.month
df["dayofweek"] = df["datetime"].dt.dayofweek

df.head()

In [ ]:
# Tạo đặc trưng quá khứ
for col in ["precipitation", "relative_humidity_2m", "cloud_cover", "temperature_2m"]:
    df[f"{col}_lag_1h"] = df[col].shift(1)
    df[f"{col}_lag_2h"] = df[col].shift(2)
    df[f"{col}_lag_3h"] = df[col].shift(3)

df["precipitation_rolling_3h"] = df["precipitation"].rolling(3).sum()
df["precipitation_rolling_6h"] = df["precipitation"].rolling(6).sum()

df.head()

In [ ]:
# Tạo nhãn mưa sau 1, 2, 3 giờ
rain_threshold = 0.1

df["precipitation_next_1h"] = df["precipitation"].shift(-1)
df["precipitation_next_2h"] = df["precipitation"].shift(-2)
df["precipitation_next_3h"] = df["precipitation"].shift(-3)

df = df.dropna().reset_index(drop=True)
df["rain_next_1h"] = (df["precipitation_next_1h"] >= rain_threshold).astype(int)
df["rain_next_2h"] = (df["precipitation_next_2h"] >= rain_threshold).astype(int)
df["rain_next_3h"] = (df["precipitation_next_3h"] >= rain_threshold).astype(int)
df = df.drop(columns=["precipitation_next_1h", "precipitation_next_2h", "precipitation_next_3h"])

df[["datetime", "precipitation", "rain_next_1h", "rain_next_2h", "rain_next_3h"]].head()

In [ ]:
# Chia train, valid, test theo thời gian
train_df = df[df["datetime"] < "2024-01-01"].copy()
valid_df = df[(df["datetime"] >= "2024-01-01") & (df["datetime"] < "2025-01-01")].copy()
test_df = df[df["datetime"] >= "2025-01-01"].copy()

df.to_csv(processed_dir / "hanoi_weather_processed.csv", index=False)
train_df.to_csv(processed_dir / "train.csv", index=False)
valid_df.to_csv(processed_dir / "valid.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Processed:", df.shape)
print("Train:", train_df.shape, train_df["datetime"].min(), train_df["datetime"].max())
print("Valid:", valid_df.shape, valid_df["datetime"].min(), valid_df["datetime"].max())
print("Test:", test_df.shape, test_df["datetime"].min(), test_df["datetime"].max())